In [1]:

import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import pulp
import pickle
import xgboost as xgb
from db_connector import list_all_tables, load_table_to_df
import warnings
warnings.filterwarnings('ignore')

# 1. 단 한 번의 데이터 로드
print("1. DB 데이터 로딩 중...")
all_tables = list_all_tables()
required_tables = ['user_profiles', 'meal_candidates', 'meal_candidate_items', 'foods', 
                   'food_logs', 'recommendation_candidates', 'recommendation_runs', 
                   'user_item_interactions', 'user_allergens', 'food_allergens']
all_dfs = {table: load_table_to_df(table) for table in all_tables if table in required_tables}
print("  -> 데이터 로드 완료!")

# 2. 모델 로드
print("2. 사전 학습된 추천 모델 로딩 중...")
with open('lightfm_model.pkl', 'rb') as f:
    lightfm_data = pickle.load(f)

xgb_model = xgb.XGBClassifier()
xgb_model.load_model('xgboost_model.json')
print("  -> 모델 로드 완료!")


1. DB 데이터 로딩 중...
  -> 데이터 로드 완료!
2. 사전 학습된 추천 모델 로딩 중...
  -> 모델 로드 완료!


In [2]:

def get_meal_conditions(run_id, all_dfs):
    df = all_dfs.get('recommendation_runs')
    if df is None: return None, None, None
    try:
        run_data = df[df['run_id'] == int(run_id)]
    except:
        run_data = df[df['run_id'] == run_id]
    if run_data.empty: return None, None, None
    row = run_data.iloc[0]
    return row['user_id'], row['target_meal_calories_kcal'], row['target_meal_budget_krw']

def generate_meal_candidates(user_id, target_meal_calories_kcal, target_meal_budget_krw, all_dfs):
    foods_df = all_dfs['foods']
    user_prof_df = all_dfs['user_profiles']
    user_profile = user_prof_df[user_prof_df['user_id'] == user_id].iloc[0]
    goal_type = user_profile['goal_type']
    
    user_al = all_dfs.get('user_allergens', pd.DataFrame())
    food_al = all_dfs.get('food_allergens', pd.DataFrame())
    forbidden_foods = []
    if not user_al.empty and not food_al.empty:
        my_allergens = user_al[user_al['user_id'] == user_id]['allergen_id'].tolist()
        forbidden_foods = food_al[food_al['allergen_id'].isin(my_allergens)]['food_id'].tolist()
        
    meal_cal = target_meal_calories_kcal / 3
    if goal_type == 'bulk':
        min_cal, max_cal = meal_cal + 300, meal_cal + 500
        ratios = {'carbs': 0.5, 'protein': 0.3, 'fat': 0.2}
    elif goal_type == 'diet':
        min_cal, max_cal = meal_cal - 500, meal_cal - 300
        ratios = {'carbs': 0.4, 'protein': 0.4, 'fat': 0.2}
    else:
        min_cal, max_cal = meal_cal * 0.9, meal_cal * 1.1
        ratios = {'carbs': 0.5, 'protein': 0.2, 'fat': 0.3}
        
    prob = pulp.LpProblem("Meal_Gen", pulp.LpMinimize)
    food_items = [f for f in foods_df.to_dict('records') 
                  if f['price_krw'] <= target_meal_budget_krw and f['food_id'] not in forbidden_foods]
                  
    food_vars = pulp.LpVariable.dicts("food", [f['food_id'] for f in food_items], 0, 1, pulp.LpBinary)
    prob += pulp.lpSum([f['price_krw'] * food_vars[f['food_id']] for f in food_items])
    prob += pulp.lpSum([food_vars[f['food_id']] for f in food_items]) >= 1
    prob += pulp.lpSum([food_vars[f['food_id']] for f in food_items]) <= 3
    prob += pulp.lpSum([f['price_krw'] * food_vars[f['food_id']] for f in food_items]) <= target_meal_budget_krw
    prob += pulp.lpSum([f['calories_kcal'] * food_vars[f['food_id']] for f in food_items]) >= min_cal
    prob += pulp.lpSum([f['calories_kcal'] * food_vars[f['food_id']] for f in food_items]) <= max_cal
    
    for macro, ratio in ratios.items():
        target_g = (meal_cal * ratio) / (9 if macro == 'fat' else 4)
        prob += pulp.lpSum([f[macro+'_g'] * food_vars[f['food_id']] for f in food_items]) >= target_g * 0.7
        prob += pulp.lpSum([f[macro+'_g'] * food_vars[f['food_id']] for f in food_items]) <= target_g * 1.3
        
    next_cand_id = int(all_dfs['meal_candidates']['candidate_id'].max() + 1) if not all_dfs['meal_candidates'].empty else 1
    next_item_id = int(all_dfs['meal_candidate_items']['meal_candidate_item_id'].max() + 1) if not all_dfs['meal_candidate_items'].empty else 1

    meal_candidates = []
    meal_candidate_items = []
    now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for _ in range(5):  # 빠른 시연을 위해 5개만 추출
        prob.solve(pulp.PULP_CBC_CMD(msg=0))
        if pulp.LpStatus[prob.status] == 'Optimal':
            selected_foods = [f for f in food_items if food_vars[f['food_id']].varValue == 1]
            cand_id = next_cand_id
            next_cand_id += 1
            
            t_price = sum(f['price_krw'] for f in selected_foods)
            t_cal = sum(f['calories_kcal'] for f in selected_foods)
            t_protein = sum(f['protein_g'] for f in selected_foods)
            t_fat = sum(f['fat_g'] for f in selected_foods)
            t_carbs = sum(f['carbs_g'] for f in selected_foods)
            
            food_ids_sorted = sorted([str(f['food_id']) for f in selected_foods])
            food_names_sorted = all_dfs['foods'].set_index('food_id')['food_name'].to_dict()
            
            meal_candidates.append({
                'candidate_id': cand_id, 'candidate_name': ", ".join(food_names_sorted[int(food_id)] for food_id in food_ids_sorted),
                'candidate_fingerprint': None, 'fingerprint_version': None,
                'meal_type': None, 'meal_channel': None,
                'total_price_krw': t_price, 'total_calories_kcal': t_cal,
                'total_protein_g': t_protein, 'total_fat_g': t_fat, 'total_carbs_g': t_carbs,
                'generation_source': 'milp_generated', 'is_active': True,
                'created_at': now_str, 'updated_at': now_str
            })
            
            for idx, f in enumerate(selected_foods, 1):
                meal_candidate_items.append({
                    'meal_candidate_item_id': next_item_id, 'candidate_id': cand_id,
                    'food_id': f['food_id'], 'quantity_g': None,
                    'quantity_label': None, 'quantity_bucket': None,
                    'item_order': idx, 'item_price_krw': f['price_krw'],
                    'item_calories_kcal': f['calories_kcal'], 'item_protein_g': f['protein_g'],
                    'item_fat_g': f['fat_g'], 'item_carbs_g': f['carbs_g']
                })
                next_item_id += 1
            prob += pulp.lpSum([food_vars[f['food_id']] for f in selected_foods]) <= len(selected_foods) - 1
        else:
            break
    return meal_candidates, meal_candidate_items


In [3]:

def calculate_lightfm_scores(user_id, meal_candidates, lightfm_data):
    model = lightfm_data['model']
    dataset = lightfm_data['dataset']
    user_map = dataset.mapping()[0]
    item_map = dataset.mapping()[2]
    
    scores = {}
    if user_id not in user_map:
        return {c['candidate_id']: 0.5 for c in meal_candidates}
        
    uid_mapped = user_map[user_id]
    for cand in meal_candidates:
        cid = cand['candidate_id']
        if cid in item_map:
            pred = model.predict(np.array([uid_mapped]), np.array([item_map[cid]]))[0]
            pred = 1 / (1 + np.exp(-pred)) # Sigmoid 정규화
        else:
            pred = 0.5 # Cold start
        scores[cid] = pred
    return scores

def calculate_xgboost_probabilities(user_id, meal_candidates, meal_candidate_items, all_dfs, xgb_model):
    profiles = all_dfs['user_profiles']
    foods = all_dfs['foods']
    
    cand_df = pd.DataFrame(meal_candidates)
    items_df = pd.DataFrame(meal_candidate_items)
    if cand_df.empty: return {}

    user_prof = profiles[profiles['user_id'] == user_id].copy()
    if user_prof.empty: return {c['candidate_id']: 0.5 for c in meal_candidates}
        
    user_prof['goal_type_enc'] = user_prof['goal_type'].map({'diet': 0, 'bulk': 1, 'maintain': 2}).fillna(-1)
    user_prof['sex_enc'] = user_prof['sex'].map({'M': 0, 'F': 1}).fillna(-1)
    
    full_df = cand_df[['candidate_id', 'total_price_krw', 'total_calories_kcal', 'total_protein_g', 'total_fat_g', 'total_carbs_g']].copy()
    full_df['user_id'] = user_id
    full_df = full_df.merge(user_prof[['user_id', 'age_years_snapshot', 'activity_factor', 'goal_type_enc', 'sex_enc']], on='user_id', how='left')
    
    food_stats = items_df.merge(foods[['food_id', 'is_low_fat', 'is_high_protein']], on='food_id')
    meal_stats = food_stats.groupby('candidate_id').agg({'is_low_fat': 'sum', 'is_high_protein': 'sum'}).reset_index()
    meal_stats.columns = ['candidate_id', 'count_low_fat', 'count_high_protein']
    
    full_df = full_df.merge(meal_stats, on='candidate_id', how='left').fillna(0)
    
    feature_cols = ['age_years_snapshot', 'activity_factor', 'goal_type_enc', 'sex_enc', 
                    'total_price_krw', 'total_calories_kcal', 'total_protein_g', 'total_fat_g', 'total_carbs_g',
                    'count_low_fat', 'count_high_protein']
    
    X_infer = full_df[feature_cols]
    probs = xgb_model.predict_proba(X_infer)[:, 1]
    
    return dict(zip(full_df['candidate_id'], probs))

def calculate_mmr_penalties(user_id, meal_candidates, meal_candidate_items, all_dfs, current_date=None):
    food_logs = all_dfs.get('food_logs', pd.DataFrame())
    rec_cands = all_dfs.get('recommendation_candidates', pd.DataFrame())

    items_df = pd.DataFrame(meal_candidate_items)
    if food_logs.empty: return {c['candidate_id']: (0.0, 0.0) for c in meal_candidates}

    user_logs = food_logs[food_logs['user_id'] == user_id].copy()
    user_logs['consumed_at'] = pd.to_datetime(user_logs['consumed_at'])
    current_date = user_logs['consumed_at'].max() if not user_logs.empty else datetime.datetime.now()
    week_ago = current_date - timedelta(days=7)
    recent_logs = user_logs[user_logs['consumed_at'] >= week_ago]
    food_counts = recent_logs['food_id'].value_counts().to_dict()

    if not rec_cands.empty and 'recommendation_candidate_id' in recent_logs.columns:
        merged_logs = recent_logs.merge(rec_cands[['recommendation_candidate_id', 'candidate_id']], on='recommendation_candidate_id', how='inner')
        candidate_counts = merged_logs['candidate_id'].value_counts().to_dict()
    else:
        candidate_counts = {}

    CANDIDATE_PENALTY_WEIGHT = 0.15
    FOOD_PENALTY_WEIGHT = 0.05
    penalties = {}
    
    for cand in meal_candidates:
        c_id = cand['candidate_id']
        cand_penalty = candidate_counts.get(c_id, 0) * CANDIDATE_PENALTY_WEIGHT
        food_penalty = 0.0
        c_items = items_df[items_df['candidate_id'] == c_id]
        for f_id in c_items['food_id']:
            food_penalty += food_counts.get(f_id, 0) * FOOD_PENALTY_WEIGHT
        penalties[c_id] = (cand_penalty, food_penalty)
        
    return penalties


In [4]:

def process_recommendation_pipeline(run_id, all_dfs, xgb_model, lightfm_data, override_cals=None, override_budget=None):
    user_id, target_cals, budget = get_meal_conditions(run_id, all_dfs)
    if user_id is None: 
        print("User ID를 찾을 수 없습니다.")
        return None
    
    if override_cals is not None: target_cals = override_cals
    if override_budget is not None: budget = override_budget
    
    print(f"파이프라인 실행 시작 (User: {user_id}, Target Cals: {target_cals}, Budget: {budget})")
    
    # 1. 식단 생성
    meal_cands, meal_items = generate_meal_candidates(user_id, target_cals, budget, all_dfs)
    if not meal_cands:
        print("조건을 만족하는 식단을 생성할 수 없습니다.")
        return None
        
    # 2. 스코어링
    lfm_scores = calculate_lightfm_scores(user_id, meal_cands, lightfm_data)
    xgb_probs = calculate_xgboost_probabilities(user_id, meal_cands, meal_items, all_dfs, xgb_model)
    mmr_pens = calculate_mmr_penalties(user_id, meal_cands, meal_items, all_dfs)
    
    rc_data = []
    now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    existing_rc = all_dfs.get('recommendation_candidates', pd.DataFrame())
    next_rc_id = int(existing_rc['recommendation_candidate_id'].max() + 1) if not existing_rc.empty else 1
    
    for i, cand in enumerate(meal_cands):
        cid = cand['candidate_id']
        lfm_score = float(lfm_scores.get(cid, 0.5))
        xgb_prob = float(xgb_probs.get(cid, 0.5))
        mmr_p, repeat_p = mmr_pens.get(cid, (0.0, 0.0))
        
        final_score = (xgb_prob * 0.5) + (lfm_score * 0.3) - mmr_p - repeat_p
        
        rc_data.append({
            'recommendation_candidate_id': next_rc_id,
            'run_id': run_id,
            'candidate_id': cid,
            'milp_feasible': True,
            'milp_rank': i + 1,
            'rule_score': 0.0,
            'lightfm_score': lfm_score,
            'xgboost_probability': xgb_prob,
            'mmr_penalty': float(mmr_p),
            'repeat_food_penalty': float(repeat_p),
            'repeat_combo_penalty': 0.0,
            'final_score': float(final_score),
            'final_rank': None,
            'feature_snapshot': None,
            'score_breakdown': None,
            'was_selected': None,
            'selected_at': None,
            'created_at': pd.to_datetime(now_str)
        })
        next_rc_id += 1
    
    rc_data.sort(key=lambda x: x['final_score'], reverse=True)
    for rank, item in enumerate(rc_data, 1):
        item['final_rank'] = rank
        
    final_df = pd.DataFrame(rc_data)
    if not existing_rc.empty:
        final_df = final_df[existing_rc.columns]
        
    return final_df


In [5]:

# 파이프라인 실행
final_result_df = process_recommendation_pipeline(
    run_id=50005, 
    all_dfs=all_dfs, 
    xgb_model=xgb_model, 
    lightfm_data=lightfm_data,
    override_cals=2174,     # recommendation.ipynb 에서 직접 입력했던 값 반영
    override_budget=13000   # recommendation.ipynb 에서 직접 입력했던 값 반영
)

if final_result_df is not None:
    print("최종 추천 후보 결과 DataFrame:")
    display(final_result_df)


파이프라인 실행 시작 (User: 1, Target Cals: 2174, Budget: 13000)
최종 추천 후보 결과 DataFrame:


,recommendation_candidate_id,run_id,candidate_id,milp_feasible,milp_rank,rule_score,lightfm_score,xgboost_probability,mmr_penalty,repeat_food_penalty,repeat_combo_penalty,final_score,final_rank,feature_snapshot,score_breakdown,was_selected,selected_at,created_at
0,55043,50005,50075,True,4,0.0,0.5,0.523993,0.0,0.0,0.0,0.411996,1,None,None,False,None,2026-06-05 20:29:05
1,55040,50005,50072,True,1,0.0,0.5,0.458467,0.0,0.0,0.0,0.379234,2,None,None,False,None,2026-06-05 20:29:05
2,55044,50005,50076,True,5,0.0,0.5,0.431048,0.0,0.0,0.0,0.365524,3,None,None,False,None,2026-06-05 20:29:05
3,55042,50005,50074,True,3,0.0,0.5,0.388967,0.0,0.0,0.0,0.344483,4,None,None,False,None,2026-06-05 20:29:05
4,55041,50005,50073,True,2,0.0,0.5,0.375287,0.0,0.0,0.0,0.337644,5,None,None,False,None,2026-06-05 20:29:05
